# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [242]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [243]:
# get file path
file_path="./data/AviationData.csv"

# load data
df =pd.read_csv(file_path,encoding="latin_1")

# get first five rows
df.head()

/var/folders/_r/g1b4b9xn2nn0xtb2011g11v80000gp/T/ipykernel_80273/421208516.py:5: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df =pd.read_csv(file_path,encoding="latin_1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [244]:
# get missing values per column
df.isna().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [245]:
# get column data types
df.dtypes

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                   object
Longitude                  object
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [246]:
# get statistical summary
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [284]:
df =df[(df["Event.Date"]>"1982-12-31") & (df["Investigation.Type"]=="Accident") & (df['Amateur.Built']=="No")]

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [248]:
# cols that represent injury or destruction
injury_des_cols=["Total.Fatal.Injuries",
"Total.Serious.Injuries",
"Total.Minor.Injuries",
"Total.Uninjured","Injury.Severity","Aircraft.damage"
]
# get first five rows
df[injury_des_cols].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Injury.Severity,Aircraft.damage


In [249]:
# get unique values in these columns
df["Total.Fatal.Injuries"].unique()
df["Total.Serious.Injuries"].unique()
df["Total.Minor.Injuries"].unique()
df["Total.Uninjured"].unique()
df["Injury.Severity"].unique()


<StringArray>
[]
Length: 0, dtype: str

In [250]:
# statistical summary of injury and destruction column
df[injury_des_cols].describe()


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN


### Decision
Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured based  on  statistical summary are heavy skewed so median imputation would label unknown cases as no injuries  and bias the data  thus decided to remove nan as nan


In [251]:
# calculate total people on the aircraft
df["Total.People"]=df["Total.Uninjured"]+df["Total.Serious.Injuries"]+df["Total.Fatal.Injuries"]+df["Total.Minor.Injuries"]

df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People


**Handling missing injury counts**

The `Total.People` column is calculated by adding the fatal, serious, minor, and uninjured counts for each accident. If any of these values is missing (`NaN`), the total will also be `NaN`. Consequently, the injury percentage cannot be calculated for that accident because the total number of people involved is unknown. I will retain these missing values rather than assume that they represent zero.

In [252]:
# calculate percentage of serious and fatal injuries to total
df["pct_fatal_and_serious"]=(df["Total.Serious.Injuries"]+df["Total.Fatal.Injuries"]) *100/df["Total.People"]


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

Aircraft.damage column has both unknown and nan.  I replaced 'Unknown' with NaN to standardize missing data

In [253]:
# check for unique values in aircraft.damage
df["Aircraft.damage"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [254]:
df["Aircraft.damage"] =(df["Aircraft.damage"].str.strip().str.title().replace("Unknown",np.nan))

In [255]:

destroyed_map={ "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0}

# create a column to check if aircraft is destroyed or not
df["is_destroyed"]=df["Aircraft.damage"].map(destroyed_map)


is_destroyed column is create where destroyed is mapped to 1 while substancial and minor  are counted as not destroyed thus represented as 0 and nan remain as nan

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [256]:

# general display make column to get an idea how it look like
df["Make"]

Series([], Name: Make, dtype: str)

In [257]:
# get unique values
df["Make"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [258]:
# display data type
df["Make"].dtype

<StringDtype(storage='python', na_value=nan)>

In [259]:
df["Make"].unique()

<StringArray>
[]
Length: 0, dtype: str

#### cleaning task
1. standardize the casing there is mixture of upper,lower,title

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [260]:
# standardize the casing and remove white spaces
df["Make"]=df["Make"].str.strip().str.title()

In [261]:
# get total of missing rows
df["Model"].isna().sum()

np.int64(0)

In [262]:
# drop rows with missing model
df.dropna(subset=("Model"),inplace=True)


In [263]:
# counts for each Make and Model, use
df.groupby(["Make", "Model"]).size().sort_values(ascending=False)

Series([], dtype: int64)

#### Conclusion
Model labels are not unique to each make

In [264]:
# create plane.type column that unique indentify plane base on make and model
df["Plane.Type"]=df['Make'] + ' ' + df['Model']
df["Plane.Type"]
df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,pct_fatal_and_serious,is_destroyed,Plane.Type


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [265]:
cols=[
"Engine.Type",
"Weather.Condition",
"Number.of.Engines",

"Purpose.of.flight",
"Broad.phase.of.flight"]

In [266]:
# display first five columns
df[cols].head(10)

,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight


In [267]:
# get summary of the cols
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Engine.Type            0 non-null      str    
 1   Weather.Condition      0 non-null      str    
 2   Number.of.Engines      0 non-null      float64
 3   Purpose.of.flight      0 non-null      str    
 4   Broad.phase.of.flight  0 non-null      str    
dtypes: float64(1), str(4)
memory usage: 132.0 bytes


In [268]:
# get unique engine type
df["Engine.Type"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [269]:
# standardize the engine type column
df["Engine.Type"]=df["Engine.Type"].str.strip().str.title().replace({'Unknown':np.nan,'None':np.nan,'Unk':np.nan})

standardize Engine.Type column by replacing unknown,unk,none  to nan

In [270]:
# get unique weather condition
df["Weather.Condition"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [271]:
# standardize the weather condition column
df["Weather.Condition"]=df["Weather.Condition"].str.strip().str.upper().replace({'UNK':np.nan})

standardized Weather.condition column by replacing unk to nan

In [272]:
# get unique number of engines condition
df["Number.of.Engines"].unique()

array([], dtype=float64)

In [273]:
# get unique values purpose of flight condition
df["Purpose.of.flight"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [274]:
# standardize purpose of flight
df["Purpose.of.flight"]=df["Purpose.of.flight"].str.strip().replace('Unknown',np.nan)

In [275]:
df["Broad.phase.of.flight"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [276]:
#standardize broad phase of flight
df["Broad.phase.of.flight"]=df["Broad.phase.of.flight"].str.strip().replace('Unknown',np.nan)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [277]:
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,pct_fatal_and_serious,is_destroyed,Plane.Type


In [278]:
# calculate percentage of missing values
(df.isna().sum() * 100/len(df)).sort_values(ascending=False)

Event.Id                 NaN
Investigation.Type       NaN
Accident.Number          NaN
Event.Date               NaN
Location                 NaN
Country                  NaN
Latitude                 NaN
Longitude                NaN
Airport.Code             NaN
Airport.Name             NaN
Injury.Severity          NaN
Aircraft.damage          NaN
Aircraft.Category        NaN
Registration.Number      NaN
Make                     NaN
Model                    NaN
Amateur.Built            NaN
Number.of.Engines        NaN
Engine.Type              NaN
FAR.Description          NaN
Schedule                 NaN
Purpose.of.flight        NaN
Air.carrier              NaN
Total.Fatal.Injuries     NaN
Total.Serious.Injuries   NaN
Total.Minor.Injuries     NaN
Total.Uninjured          NaN
Weather.Condition        NaN
Broad.phase.of.flight    NaN
Report.Status            NaN
Publication.Date         NaN
Total.People             NaN
pct_fatal_and_serious    NaN
is_destroyed             NaN
Plane.Type    

In [279]:
# drop coliumns that have more than 50% missing values
drop_cols=["Schedule",                 
"Air.carrier" ,             
"FAR.Description",         
"Aircraft.Category" ,       
"Longitude" ,             
"Latitude"]   

In [280]:
# drop columns

cleaned_df=df.drop(columns=drop_cols)

In [281]:
# confirm columns have been dropped
cleaned_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Airport.Code', 'Airport.Name',
       'Injury.Severity', 'Aircraft.damage', 'Registration.Number', 'Make',
       'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type',
       'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries',
       'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition',
       'Broad.phase.of.flight', 'Report.Status', 'Publication.Date',
       'Total.People', 'pct_fatal_and_serious', 'is_destroyed', 'Plane.Type'],
      dtype='str')

#### decision
Dropped columns that have more than 50% data missing since imputing it is like inventing data

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [282]:
cleaned_df.to_csv("./data/cleaned_aviation_data.csv",index=False)

In [283]:
df["Amateur.Built"]

Series([], Name: Amateur.Built, dtype: str)